In [48]:
import os
os.environ.pop("PYTHONPATH", None)

!pip install llama-index==0.12.2 llama-index-llms-openai==0.3.1 \
    llama-index-embeddings-huggingface llama-index-readers-wikipedia==0.3.0 wikipedia==1.4.0 \
    openai==1.55.3 trulens==1.2.0 trulens-providers-openai==1.2.0 \
    langchain nltk>=3.8.1 \
    streamlit==1.35.0 watchdog kubernetes==26.1.0 \
    packaging>=24.0 --quiet

In [49]:
import os
import openai

os.environ["OPENAI_API_KEY"] = "sk-..." #copy your api key


In [50]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, PromptTemplate, Document, get_response_synthesizer
from openai import OpenAI

In [51]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt"
urllib.request.urlretrieve(url, "data/paul_graham_essay.txt")

('data/paul_graham_essay.txt', <http.client.HTTPMessage at 0x1b141834490>)

In [52]:
doc = SimpleDirectoryReader("data").load_data()

In [53]:
index = VectorStoreIndex.from_documents(doc)
index

In [54]:
query_engine = index.as_query_engine()
response = query_engine.query("Who is the author of this article?")
print(response)

Paul Graham


In [55]:
def generate_answer(question):
    message = [
        {"role": "system",
         "content": "You are a helpful assistant."},
        {"role": "user",
         "content" : question}
    ]
    response = openai.chat.completions.create(model='gpt-3.5-turbo', messages=message)
    return response.choices[0].message.content

In [56]:
llm_response = generate_answer("What is the family city of Dresden in Korea?")
print(llm_response)

The family city of Dresden in Korea is Busan. This sisterhood relationship was established to promote cultural exchange and cooperation between the two cities.


In [57]:
retriever = index.as_retriever()
retrieve_result = retriever.retrieve("Who is the author of this article?")
context = []
for ret in retrieve_result:
    context.append(ret.text)
context[0]

'After I moved to New York I became her de facto studio assistant.\n\nShe liked to paint on big, square canvases, 4 to 5 feet on a side. One day in late 1994 as I was stretching one of these monsters there was something on the radio about a famous fund manager. He wasn\'t that much older than me, and was super rich. The thought suddenly occurred to me: why don\'t I become rich? Then I\'ll be able to work on whatever I want.\n\nMeanwhile I\'d been hearing more and more about this new thing called the World Wide Web. Robert Morris showed it to me when I visited him in Cambridge, where he was now in grad school at Harvard. It seemed to me that the web would be a big deal. I\'d seen what graphical user interfaces had done for the popularity of microcomputers. It seemed like the web would do the same for the internet.\n\nIf I wanted to get rich, here was the next train leaving the station. I was right about that part. What I got wrong was the idea. I decided we should start a company to put

In [58]:
question = f"""
Given the Context and not prior knowledge, answer the query and the reasons of the answer.
###Context : {context}
###Query : Who is the author of this article?
###Answer : 
"""
llm_response = generate_answer(question)
llm_response

"The author of this article is Paul Graham. \n\nThe reasons for this answer are:\n1. The article mentions details about Paul Graham's experiences, such as moving to New York, starting a company called Viaweb, and his involvement with Y Combinator.\n2. The article also discusses Paul Graham's personal life, relationships, and professional journey, which align with what is known about Paul Graham's background and career."

In [59]:
question = "What are the temperature and air pressure conditions under which natural diamonds are produced?"
res = query_engine.query(question)
print(res)

Natural diamonds are produced deep within the Earth's mantle under high temperature and high pressure conditions.


In [60]:
new_text = "Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."

new_doc = Document(text=new_text, id_="new_doc_id")
index.insert(new_doc)

new_query_engine = index.as_query_engine()
response = new_query_engine.query(question)
print(response)

Natural diamonds are formed in metallic melts at temperatures ranging from 900 to 1,400 degrees Celsius and at pressures of 5 to 6 gigapascals.


In [61]:
low_temp = 5000
high_temp = 10000
updated_text = f"Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of {low_temp}–{high_temp} °C and at pressures of 5–6 GPa."
print(updated_text)

new_doc.set_content(value=updated_text)
output = index.update_ref_doc(new_doc, update_kwargs={"delete_kwargs" : True})
print(output)

updated_query_engine = index.as_query_engine()
response = updated_query_engine.query(question)
print(response)

Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 5000–10000 °C and at pressures of 5–6 GPa.
None
Natural diamonds are formed in metallic melts at temperatures ranging from 5000 to 10000 °C and at pressures of 5 to 6 GPa.


In [62]:
low_temp = 10
high_temp = 500
refresh_text = f"Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of {low_temp}–{high_temp} °C and at pressures of 5–6 GPa."
print(refresh_text)

new_doc.set_content(value=refresh_text)
output = index.refresh_ref_docs([new_doc])
print(output)

refresh_query_engine = index.as_query_engine()
response = refresh_query_engine.query(question)
print(response)

Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 10–500 °C and at pressures of 5–6 GPa.
[True]
Natural diamonds are formed in metallic melts at temperatures ranging from 10–500 °C and at pressures of 5–6 GPa.


In [63]:
index.delete( doc_id="new_doc_id")
delete_query_engine = index.as_query_engine()
response = delete_query_engine.query(question)
print(response)

Natural diamonds are produced deep within the Earth's mantle under high temperature and high pressure conditions.


### RAG

In [64]:
import textwrap

def pretty_print(answer):
    wrapped_text = textwrap.fill(answer, width=80)
    return wrapped_text

In [66]:
city_question = 'Which Korean city has a relationship with Dresden?'

kr_city = generate_answer(city_question)
print("Answer: ", pretty_print(kr_city))

city_related_to_question = 'Dresden'

Answer:  The Korean city of Gyeongju has a sister city relationship with Dresden,
Germany. Sister city relationships are established to promote cultural and
economic exchanges between cities around the world.


In [67]:
city_name_path = r'city_short.txt' #change this path

city_names = []

with open(city_name_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()
    for line in lines:
        city = line.split(':')[0][:-1]
        city_names.append(city)

print(city_names) #100 different cities

if city_related_to_question not in city_names:
  city_names.append(city_related_to_question)

['New York City', 'London', 'Paris', 'Tokyo', 'Berlin', 'Shanghai', 'Los Angeles', 'Istanbul', 'Sydney', 'Seoul', 'Saint Petersburg', 'Mumbai', 'Beijing', 'Leipzig', 'Toronto', 'Rio de Janeiro', 'Mexico City', 'Budapest', 'Barcelona', 'Moscow', 'Vancouver', 'Dublin', 'Antananarivo', 'Suwon']


In [68]:
from llama_index.readers.wikipedia import WikipediaReader
import wikipedia

wikipedia.set_user_agent("my-rag-project (test@example.com)")
reader = WikipediaReader()
documents = reader.load_data(city_names, auto_suggest=False)

In [22]:
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

In [77]:
berlin_question = "What's the arts and culture scene in Berlin?"
response = query_engine.query(berlin_question)
print(response)

Berlin has a vibrant arts and culture scene with numerous galleries, museums, theaters, and music venues. The city is known for its diverse and innovative artistic community, with a rich history of creativity and expression. Berlin is home to a wide range of cultural events, festivals, and exhibitions that attract artists and art enthusiasts from around the world.


In [24]:
unknown_question_xinjiang = "What is the magnitude of the earthquake that occurred in Xinjiang, China in 2024?"
unknown_question_jaipur = "Where in Jaipur is the proposed location for the third largest cricket stadium in the world, which can accommodate about 75,000 people?"

response_xinjiang = query_engine.query(unknown_question_xinjiang)
response_jaipur = query_engine.query(unknown_question_jaipur)

print(pretty_print(response_xinjiang.response))
print("\n-------------------------\n")
print(pretty_print(response_jaipur.response))

The magnitude of the earthquake that occurred in Xinjiang, China in 2024 is not
provided in the context information.

-------------------------

Jaipur's proposed location for the third largest cricket stadium in the world,
which can accommodate about 75,000 people, is in the outskirts of the city near
the Chonp village.


In [78]:
class RAG_from_scratch:
    def retriever(self, query: str) -> list:
        response = query_engine.query(query)
        return response
        
    def generate_answer(self, query: str, context_str: list) -> str:
        message = [
            {"role": "system",
             "content": "You are a helpful assistant."},
            {"role": "user",
             "content" : 
             "Given the context information and not prior knowledge, answer the query"
             f"###Context : {context_str}"
             f"###Query : {query}"}
        ]

        response = openai.chat.completions.create( model='gpt-3.5-turbo', temperature=0, messages=message)
        return response.choices[0].message.content

    def query(self, query: str) -> str:
        context_str = self.retriever(query)
        response = self.generate_answer(query, context_str)
        return response

In [79]:
rag = RAG_from_scratch()

In [80]:
response = rag.query(berlin_question)
print(response)

Berlin has a vibrant arts and culture scene with numerous galleries, museums, theaters, and music venues. The city is known for its diverse and innovative artistic community, with a rich history of creativity and expression. Berlin is home to a wide range of cultural events, festivals, and exhibitions that attract artists and art enthusiasts from around the world.


In [71]:
response_xinjiang = rag.query(unknown_question_xinjiang)
response_jaipur = rag.query(unknown_question_jaipur)

print(response_xinjiang)
print("\n-------------------------\n")
print(response_jaipur)

I'm sorry, but based on the context provided, I am unable to provide information on the magnitude of the earthquake that occurred in Xinjiang, China in 2024.

-------------------------

The proposed location for the third largest cricket stadium in the world, which can accommodate about 75,000 people, is in Jaipur.


### Custom Query Engine

In [72]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.response_synthesizers import BaseSynthesizer


class StandardQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        response_obj = self.response_synthesizer.synthesize(query_str, nodes)
        return response_obj

In [81]:
from llama_index.core import get_response_synthesizer

retriever = index.as_retriever()
sythersizer = get_response_synthesizer()
std_quary_engine = StandardQueryEngine(retriever=retriever, response_synthesizer=sythersizer)
response = std_quary_engine.query(berlin_question)
print(response)

Berlin has a vibrant arts and culture scene with numerous galleries, museums, theaters, and music venues. The city is known for its diverse and avant-garde art scene, with many artists and creatives flocking to Berlin for its creative atmosphere. Additionally, Berlin hosts various cultural events, festivals, and exhibitions throughout the year, making it a hub for artistic expression and innovation.


In [82]:
response_xinjiang = std_quary_engine.query(unknown_question_xinjiang)
response_jaipur = std_quary_engine.query(unknown_question_jaipur)

print(response_xinjiang)
print("\n-------------------------\n")
print(response_jaipur)

I'm unable to provide information on the magnitude of the earthquake that occurred in Xinjiang, China in 2024 based on the context provided.

-------------------------

The proposed location for the third largest cricket stadium in the world, which can accommodate about 75,000 people, is in Jaipur.


In [83]:
class OurCustomQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever

    def custom_query(self, query: str):
        nodes = self.retriever.retrieve(query)
        message = f"""
        Given the context information and not prior knowledge, answer the question
        ### Context : {nodes}
        ### Question : {query}
        """
        prompt = [
            {"role" : "system",
             "content" : "You are a helpful assistant"},
            {"role" : "user",
             "content" : message}
        ]
        
        response = openai.chat.completions.create(model="gpt-3.5-turbo", temperature=0, messages=prompt)
        return response.choices[0].message.content

retriever = index.as_retriever()

custom_query_engine = OurCustomQueryEngine(retriever=retriever)        
response = custom_query_engine.query(berlin_question)
print(response)

I'm sorry, but I cannot provide an answer to the question about the arts and culture scene in Berlin based on the context information provided. If you have any other questions or need assistance with the given context, feel free to ask!
